In [ ]:
import os
import sys
import pandas as pd
import uuid

from pathlib import Path
from from_root import from_root

sys.path.insert(0, str(from_root("src")))

from model_loading import load_model
from read_and_write_docs import read_txt, read_rds, read_jsonl, read_excel_sheets
from n_gram_tracing import (
    common_ngrams,
    filter_ngrams_with_outside_occurrences_in_both_texts,
    tokenize_to_tokens,
    find_all_token_ngram_spans,
    find_independent_token_ngram_spans,
    ngram_occurrence_stats
)
from utils import apply_temp_doc_id, build_metadata_df

In [125]:
# --- set your args (strings) ---
base_loc = "/Volumes/BCross"

if not os.path.exists(base_loc):
    print("Using Offline Location")
    base_loc = "/Users/user/Documents/uni_work_offline"
else:
    print("Using Volume Location")

CORPUS = "Wiki"
DATA_TYPE = "training"

KNOWN_LOC = f"{base_loc}/datasets/author_verification/{DATA_TYPE}/{CORPUS}/known_raw.jsonl"
UNKNOWN_LOC = f"{base_loc}/datasets/author_verification/{DATA_TYPE}/{CORPUS}/unknown_raw.jsonl"
METADATA_LOC = f"{base_loc}/datasets/author_verification/{DATA_TYPE}/metadata.rds"
MODEL_LOC = f"{base_loc}/models/gpt2"

# file_loc = '/Users/user/Downloads/-as_oxYIDXEGRxLxgd6DGA vs -AXo3Eeg-Ipqx54f8ZahLQ.xlsx'
# PROBLEM = Path(file_loc).stem
PROBLEM = "Haymaker vs HeadleyDown"

print(f"Problem: {PROBLEM}")

Using Volume Location
Problem: Haymaker vs HeadleyDown


In [126]:
tokenizer, model = load_model(MODEL_LOC)

In [127]:
known = read_jsonl(KNOWN_LOC)
known = apply_temp_doc_id(known)
    
unknown = read_jsonl(UNKNOWN_LOC)
unknown = apply_temp_doc_id(unknown)

metadata = read_rds(METADATA_LOC)
filtered_metadata = metadata[
    (metadata['corpus'] == CORPUS)
    & (metadata['problem'] ==  PROBLEM)
]

agg_metadata = build_metadata_df(filtered_metadata, known, unknown)

# -----
# Get the chosen text & metadata
# -----
    
known_author = filtered_metadata['known_author'].iloc[0]
unknown_author = filtered_metadata['unknown_author'].iloc[0]

selected_known = known[known['author'] == known_author]
selected_unknown = unknown[unknown['author'] == unknown_author]

unknown_doc = selected_unknown['doc_id'].iloc[0]
unknown_text = selected_unknown['text'].iloc[0]

num_known_problems = len(selected_known)
known_texts = selected_known['text'].tolist()
print(f"There are {num_known_problems} known texts in the problem")

There are 3 known texts in the problem


In [128]:
def dedupe_ngrams(ngrams):
    """
    Deduplicate n-grams while preserving order.
    """
    return [list(x) for x in dict.fromkeys(tuple(g) for g in ngrams)]


def sort_ngrams(ngrams):
    """
    Sort first by number of tokens, then by total character length.
    """
    return sorted(
        ngrams,
        key=lambda x: (len(x), sum(len(str(token)) for token in x))
    )

In [129]:
def global_second_pass_greatest_common(
    known_texts,
    unknown_text,
    tokenizer,
    *,
    lowercase=True,
):
    all_common = []

    # First pass: collect all pairwise common n-grams
    for known_text in known_texts:
        pair_common = common_ngrams(
            text1=known_text,
            text2=unknown_text,
            tokenizer=tokenizer,
            include_subgrams=True,
            lowercase=lowercase,
        )

        all_common.extend(pair_common)

    # Problem-level candidate list
    global_common = dedupe_ngrams(all_common)
    global_common = sort_ngrams(global_common)

    if not global_common:
        return []

    unknown_stats = ngram_occurrence_stats(
        ngrams=global_common,
        text=unknown_text,
        tokenizer=tokenizer,
        lowercase=lowercase,
    )

    known_stats_list = [
        ngram_occurrence_stats(
            ngrams=global_common,
            text=known_text,
            tokenizer=tokenizer,
            lowercase=lowercase,
        )
        for known_text in known_texts
    ]

    kept = []

    for g in dict.fromkeys(tuple(x) for x in global_common):
        unknown_keep = unknown_stats.get(g, {}).get("keep", False)

        known_keep = any(
            stats.get(g, {}).get("keep", False)
            for stats in known_stats_list
        )

        if unknown_keep and known_keep:
            kept.append(list(g))

    kept = sort_ngrams(kept)
    
    return kept

In [130]:
def effective_unknown_scored_ngrams_after_greatest_common(
    ngrams,
    unknown_text,
    tokenizer,
    *,
    lowercase=True,
):
    """
    Given a final n-gram list, return the n-grams that would still have
    at least one uncovered occurrence in the unknown document.

    This approximates the extra effect of using greatest_common=True
    inside score_ngrams_to_df.
    """
    stats = ngram_occurrence_stats(
        ngrams=ngrams,
        text=unknown_text,
        tokenizer=tokenizer,
        lowercase=lowercase,
    )

    kept = [
        row["ngram"]
        for row in stats.values()
        if row["outside_count"] > 0
    ]

    return sort_ngrams(kept)

In [131]:
global_kept_ngrams = global_second_pass_greatest_common(
    known_texts=known_texts,
    unknown_text=unknown_text,
    tokenizer=tokenizer,
    lowercase=True,
)

print(f"Global second-pass version kept: {len(global_kept_ngrams)} n-grams")

Global second-pass version kept: 143 n-grams


In [132]:

global_effective_scored_ngrams = effective_unknown_scored_ngrams_after_greatest_common(
    ngrams=global_kept_ngrams,
    unknown_text=unknown_text,
    tokenizer=tokenizer,
    lowercase=True,
)

print(f"Global candidate n-grams before scoring cleanup: {len(global_kept_ngrams)}")
print(f"Global effective n-grams after scoring cleanup: {len(global_effective_scored_ngrams)}")

Global candidate n-grams before scoring cleanup: 143
Global effective n-grams after scoring cleanup: 143


In [133]:
def compare_ngram_lists(before, after, label=""):
    before_set = set(tuple(x) for x in before)
    after_set = set(tuple(x) for x in after)

    removed = before_set - after_set
    kept = before_set & after_set

    print(f"\n{label}")
    print(f"Before: {len(before_set)}")
    print(f"After: {len(after_set)}")
    print(f"Kept: {len(kept)}")
    print(f"Removed by scoring-stage greatest_common: {len(removed)}")

    return {
        "kept": sort_ngrams([list(x) for x in kept]),
        "removed": sort_ngrams([list(x) for x in removed]),
    }

In [134]:
global_comparison = compare_ngram_lists(
    global_kept_ngrams,
    global_effective_scored_ngrams,
    label="Global second-pass method",
)


Global second-pass method
Before: 143
After: 143
Kept: 143
Removed by scoring-stage greatest_common: 0


In [135]:
global_removed = global_comparison["removed"]

print(global_removed[:20])

[]


In [136]:
import html
import uuid
from typing import Any, List, Sequence, Optional, Tuple

from IPython.display import display, HTML

from n_gram_tracing import (
    tokenize_to_tokens,
    tokens_to_text,
    find_all_token_ngram_spans,
    find_independent_token_ngram_spans,
)


_NGRAM_COLOURS = [
    "#fff3b0",
    "#c7f9cc",
    "#bde0fe",
    "#ffc8dd",
    "#d0bfff",
    "#ffd6a5",
    "#caffbf",
    "#a0c4ff",
    "#ffadad",
    "#e9edc9",
    "#bae6fd",
    "#fecdd3",
]


def _html_escape(x: Any) -> str:
    return html.escape(str(x), quote=True)


def _normalise_ngram_list(ngrams):
    """
    Accept either:
      - a single n-gram like ['Ġwe']
      - a list of n-grams like [['Ġwe'], ['Ġwe', 'Ġhave']]
    and always return a list of n-grams.
    """
    if not ngrams:
        return []

    if isinstance(ngrams[0], str):
        return [list(ngrams)]

    return [list(g) for g in ngrams]


def _ngram_colour(ngram_idx: int) -> str:
    return _NGRAM_COLOURS[ngram_idx % len(_NGRAM_COLOURS)]


def _stacked_background(colours: Sequence[str]) -> str:
    """
    Create a vertical stacked background for overlapping n-gram hits.
    """
    colours = list(dict.fromkeys(colours))

    if not colours:
        return "transparent"

    if len(colours) == 1:
        return colours[0]

    n = len(colours)
    parts = []

    for i, colour in enumerate(colours):
        start = 100 * i / n
        end = 100 * (i + 1) / n
        parts.append(f"{colour} {start:.1f}% {end:.1f}%")

    return "linear-gradient(to bottom, " + ", ".join(parts) + ")"


def _collect_ngram_spans(
    text: str,
    ngrams: Sequence[Sequence[Any]],
    *,
    tokenizer: Any,
    lowercase: bool = True,
    greatest_common: bool = False,
    all_ngrams: Optional[Sequence[Sequence[Any]]] = None,
) -> Tuple[List[Any], List[dict]]:
    """
    Return tokenized text plus span records for all requested n-grams.
    """
    tokens = tokenize_to_tokens(text, tokenizer=tokenizer, lowercase=lowercase)

    span_rows = []

    for ng_idx, ng in enumerate(ngrams):
        if greatest_common:
            if all_ngrams is None:
                raise ValueError("all_ngrams must be provided when greatest_common=True")

            spans = find_independent_token_ngram_spans(
                tokens=tokens,
                ngram_tokens=list(ng),
                all_ngrams=all_ngrams,
                allow_overlaps=True,
            )
        else:
            spans = find_all_token_ngram_spans(
                tokens=tokens,
                ngram_tokens=list(ng),
                allow_overlaps=True,
            )

        for occurrence_idx, (s, e) in enumerate(spans):
            span_rows.append({
                "start": s,
                "end": e,
                "ngram": list(ng),
                "ngram_idx": ng_idx,
                "occurrence_idx": occurrence_idx,
                "length": e - s,
            })

    return sorted(
        span_rows,
        key=lambda row: (row["start"], -row["length"], row["ngram_idx"])
    ), tokens


def _render_plain_token_html(
    token_text: str,
    token_idx: int,
    *,
    show_token_indices: bool = True,
) -> str:
    if show_token_indices:
        return f"""
        <span class="ngram-token" title="token {token_idx}">
            <span class="ngram-token-index">{token_idx}</span>
            <span class="ngram-token-text">{_html_escape(token_text)}</span>
        </span>
        """

    return f"""
    <span class="ngram-token" title="token {token_idx}">
        <span class="ngram-token-text">{_html_escape(token_text)}</span>
    </span>
    """


def _render_hit_token_html(
    token_text: str,
    token_idx: int,
    hits: List[dict],
    tokenizer: Any,
    *,
    show_token_indices: bool = True,
    show_start_labels: bool = True,
    max_hover_hits: int = 12,
) -> str:
    colours = [_ngram_colour(hit["ngram_idx"]) for hit in hits]
    background = _stacked_background(colours)

    ngram_classes = " ".join(
        f"ngram-g-{hit['ngram_idx'] + 1}"
        for hit in hits
    )

    hover_lines = []

    for hit in hits[:max_hover_hits]:
        label = tokens_to_text(hit["ngram"], tokenizer)
        hover_lines.append(
            f"G{hit['ngram_idx'] + 1}: {label} "
            f"[{hit['start']}:{hit['end']}]"
        )

    if len(hits) > max_hover_hits:
        hover_lines.append(f"... {len(hits) - max_hover_hits} more hits")

    hover_text = "\n".join(hover_lines)

    labels = []

    if show_start_labels:
        starting_hits = [
            hit for hit in hits
            if hit["start"] == token_idx
        ]

        for hit in starting_hits[:3]:
            labels.append(
                f"<span class='ngram-hit-label'>G{hit['ngram_idx'] + 1}</span>"
            )

        if len(starting_hits) > 3:
            labels.append(
                f"<span class='ngram-hit-label'>+{len(starting_hits) - 3}</span>"
            )

    label_html = "".join(labels)

    if show_token_indices:
        return f"""
        <span class="ngram-token ngram-token-hit {ngram_classes}"
              style="background:{background};"
              title="{_html_escape(hover_text)}">
            {label_html}
            <span class="ngram-token-index">{token_idx}</span>
            <span class="ngram-token-text">{_html_escape(token_text)}</span>
        </span>
        """

    return f"""
    <span class="ngram-token ngram-token-hit {ngram_classes}"
          style="background:{background};"
          title="{_html_escape(hover_text)}">
        {label_html}
        <span class="ngram-token-text">{_html_escape(token_text)}</span>
    </span>
    """


def _render_token_stream_html(
    tokens: List[Any],
    spans: List[dict],
    tokenizer: Any,
    *,
    max_tokens: Optional[int] = None,
    show_token_indices: bool = True,
    show_start_labels: bool = True,
) -> str:
    """
    Render one token stream.

    This allows overlapping n-grams. Each token is highlighted according
    to every retained n-gram that covers it.
    """
    n_tokens = len(tokens)
    render_limit = n_tokens if max_tokens is None else min(max_tokens, n_tokens)

    token_hits = [[] for _ in range(render_limit)]

    for row in spans:
        start = max(row["start"], 0)
        end = min(row["end"], render_limit)

        for idx in range(start, end):
            token_hits[idx].append(row)

    html_parts = []

    for tok_idx in range(render_limit):
        tok_text = tokens_to_text([tokens[tok_idx]], tokenizer)
        hits = token_hits[tok_idx]

        if hits:
            html_parts.append(
                _render_hit_token_html(
                    tok_text,
                    tok_idx,
                    hits,
                    tokenizer,
                    show_token_indices=show_token_indices,
                    show_start_labels=show_start_labels,
                )
            )
        else:
            html_parts.append(
                _render_plain_token_html(
                    tok_text,
                    tok_idx,
                    show_token_indices=show_token_indices,
                )
            )

    if max_tokens is not None and n_tokens > max_tokens:
        html_parts.append(
            f"""
            <span class="ngram-omitted">
                ... {n_tokens - max_tokens} tokens omitted
            </span>
            """
        )

    return "\n".join(html_parts)


def _build_ngram_legend_html(
    ngrams: Sequence[Sequence[Any]],
    tokenizer: Any,
    *,
    viz_uid: str,
    max_legend_items: Optional[int] = 200,
) -> str:
    legend_ngrams = list(ngrams)

    if max_legend_items is None:
        shown_ngrams = legend_ngrams
    else:
        shown_ngrams = legend_ngrams[:max_legend_items]

    legend_bits = [
        f"""
        <label class="ngram-legend-chip ngram-legend-clear"
               for="{viz_uid}-none">
            Clear selection
        </label>
        """
    ]

    for ng_idx, ng in enumerate(shown_ngrams):
        colour = _ngram_colour(ng_idx)
        label = tokens_to_text(ng, tokenizer)
        g_num = ng_idx + 1

        legend_bits.append(
            f"""
            <label class="ngram-legend-chip ngram-legend-g-{g_num}"
                   for="{viz_uid}-g-{g_num}"
                   style="background:{colour};"
                   title="{_html_escape(label)}">
                G{g_num}: {_html_escape(label)}
            </label>
            """
        )

    if max_legend_items is not None and len(legend_ngrams) > max_legend_items:
        legend_bits.append(
            f"""
            <span class="ngram-legend-more">
                ... {len(legend_ngrams) - max_legend_items} more n-grams not shown in legend
            </span>
            """
        )

    return f"""
    <details class="ngram-legend" open>
        <summary>Interactive n-gram legend ({len(legend_ngrams)} n-grams)</summary>
        <div class="ngram-legend-body">
            {''.join(legend_bits)}
        </div>
    </details>
    """


def _build_panel_html(
    *,
    panel_title: str,
    panel_subtitle: str,
    token_stream_html: str,
) -> str:
    return f"""
    <div class="ngram-panel">
        <div class="ngram-panel-title">
            {_html_escape(panel_title)}
            <span class="ngram-panel-subtitle">
                {panel_subtitle}
            </span>
        </div>
        <div class="ngram-token-stream">
            {token_stream_html}
        </div>
    </div>
    """


def _build_radio_inputs_html(
    *,
    viz_uid: str,
    n_visible_legend_items: int,
) -> str:
    radio_inputs = [
        f"""
        <input type="radio"
               name="{viz_uid}-selected-ngram"
               id="{viz_uid}-none"
               class="ngram-selector"
               checked>
        """
    ]

    for g_num in range(1, n_visible_legend_items + 1):
        radio_inputs.append(
            f"""
            <input type="radio"
                   name="{viz_uid}-selected-ngram"
                   id="{viz_uid}-g-{g_num}"
                   class="ngram-selector">
            """
        )

    return "\n".join(radio_inputs)


def _build_selection_css(
    *,
    viz_uid: str,
    n_visible_legend_items: int,
) -> str:
    css_rules = []

    for g_num in range(1, n_visible_legend_items + 1):
        css_rules.append(
            f"""
            #{viz_uid}-g-{g_num}:checked ~ .ngram-container .ngram-token-hit {{
                opacity: 0.18;
                filter: grayscale(0.85);
            }}

            #{viz_uid}-g-{g_num}:checked ~ .ngram-container .ngram-token-hit.ngram-g-{g_num} {{
                opacity: 1;
                filter: none;
                outline: 3px solid #111827;
                outline-offset: 2px;
                box-shadow: 0 0 0 4px rgba(250, 204, 21, 0.55);
                position: relative;
                z-index: 3;
            }}

            #{viz_uid}-g-{g_num}:checked ~ .ngram-container .ngram-legend-chip {{
                opacity: 0.35;
            }}

            #{viz_uid}-g-{g_num}:checked ~ .ngram-container .ngram-legend-g-{g_num},
            #{viz_uid}-g-{g_num}:checked ~ .ngram-container .ngram-legend-clear {{
                opacity: 1;
            }}

            #{viz_uid}-g-{g_num}:checked ~ .ngram-container .ngram-legend-g-{g_num} {{
                outline: 3px solid #111827;
                outline-offset: 2px;
                font-weight: 800;
            }}
            """
        )

    return "\n".join(css_rules)


def build_ngram_highlights_html(
    *,
    unknown_text: str,
    known_texts: Sequence[str],
    tokenizer: Any,
    ngrams,
    known_doc_ids: Optional[Sequence[Any]] = None,
    unknown_doc_id: Optional[Any] = None,
    lowercase: bool = True,
    greatest_common: bool = False,
    all_ngrams: Optional[Sequence[Sequence[Any]]] = None,
    title: str = "Common token n-gram highlights",
    max_tokens_per_text: Optional[int] = None,
    show_token_indices: bool = True,
    show_start_labels: bool = True,
    max_legend_items: Optional[int] = 200,
) -> str:
    """
    Build a GST-style HTML visualisation for n-gram highlights.

    Important:
    - ngrams is the list of n-grams you want to plot.
    - If greatest_common=True and all_ngrams is not supplied,
      all_ngrams defaults to ngrams.
    - Clicking an item in the legend highlights that n-gram across
      the unknown text and all known texts.
    """
    ngrams = _normalise_ngram_list(ngrams)

    if greatest_common and all_ngrams is None:
        all_ngrams = ngrams

    if known_doc_ids is None:
        known_doc_ids = [f"known_{i + 1}" for i in range(len(known_texts))]

    if len(known_doc_ids) != len(known_texts):
        raise ValueError("known_doc_ids must be the same length as known_texts")

    viz_uid = f"ngramviz-{uuid.uuid4().hex[:8]}"

    if max_legend_items is None:
        n_visible_legend_items = len(ngrams)
    else:
        n_visible_legend_items = min(len(ngrams), max_legend_items)

    radio_inputs_html = _build_radio_inputs_html(
        viz_uid=viz_uid,
        n_visible_legend_items=n_visible_legend_items,
    )

    selection_css = _build_selection_css(
        viz_uid=viz_uid,
        n_visible_legend_items=n_visible_legend_items,
    )

    # Unknown text
    unk_spans, unk_tokens = _collect_ngram_spans(
        unknown_text,
        ngrams,
        tokenizer=tokenizer,
        lowercase=lowercase,
        greatest_common=greatest_common,
        all_ngrams=all_ngrams,
    )

    unk_stream_html = _render_token_stream_html(
        unk_tokens,
        unk_spans,
        tokenizer,
        max_tokens=max_tokens_per_text,
        show_token_indices=show_token_indices,
        show_start_labels=show_start_labels,
    )

    unk_subtitle_bits = [
        f"tokens: {len(unk_tokens)}",
        f"occurrences: {len(unk_spans)}",
    ]

    if unknown_doc_id is not None:
        unk_subtitle_bits.insert(0, f"ID: {_html_escape(unknown_doc_id)}")

    unknown_panel = _build_panel_html(
        panel_title="Unknown text",
        panel_subtitle=" · ".join(unk_subtitle_bits),
        token_stream_html=unk_stream_html,
    )

    # Known texts
    known_panels = []
    total_known_tokens = 0
    total_known_occurrences = 0

    for known_idx, (known_text, known_doc_id) in enumerate(zip(known_texts, known_doc_ids)):
        known_spans, known_tokens = _collect_ngram_spans(
            known_text,
            ngrams,
            tokenizer=tokenizer,
            lowercase=lowercase,
            greatest_common=greatest_common,
            all_ngrams=all_ngrams,
        )

        total_known_tokens += len(known_tokens)
        total_known_occurrences += len(known_spans)

        known_stream_html = _render_token_stream_html(
            known_tokens,
            known_spans,
            tokenizer,
            max_tokens=max_tokens_per_text,
            show_token_indices=show_token_indices,
            show_start_labels=show_start_labels,
        )

        known_panels.append(
            _build_panel_html(
                panel_title=f"Known text {known_idx + 1}",
                panel_subtitle=(
                    f"ID: {_html_escape(known_doc_id)} · "
                    f"tokens: {len(known_tokens)} · "
                    f"occurrences: {len(known_spans)}"
                ),
                token_stream_html=known_stream_html,
            )
        )

    legend_html = _build_ngram_legend_html(
        ngrams,
        tokenizer,
        viz_uid=viz_uid,
        max_legend_items=max_legend_items,
    )

    total_occurrences = len(unk_spans) + total_known_occurrences

    return f"""
    {radio_inputs_html}

    <style>
        .ngram-selector {{
            display: none;
        }}

        .ngram-container {{
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
            line-height: 1.45;
            color: #222;
        }}

        .ngram-title {{
            font-size: 20px;
            font-weight: 700;
            margin-bottom: 8px;
        }}

        .ngram-meta {{
            display: flex;
            flex-wrap: wrap;
            gap: 8px;
            margin-bottom: 14px;
        }}

        .ngram-pill {{
            background: #f3f4f6;
            border: 1px solid #d1d5db;
            border-radius: 999px;
            padding: 4px 10px;
            font-size: 13px;
        }}

        .ngram-layout {{
            display: grid;
            grid-template-columns: 1fr;
            gap: 14px;
        }}

        .ngram-panel {{
            border: 1px solid #d1d5db;
            border-radius: 8px;
            overflow: hidden;
            background: white;
        }}

        .ngram-panel-title {{
            background: #f9fafb;
            border-bottom: 1px solid #d1d5db;
            padding: 8px 10px;
            font-weight: 700;
        }}

        .ngram-panel-subtitle {{
            font-weight: 400;
            color: #6b7280;
            margin-left: 8px;
            font-size: 13px;
        }}

        .ngram-token-stream {{
            padding: 10px;
            max-height: 420px;
            overflow: auto;
            font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace;
            font-size: 13px;
            line-height: 2.2;
        }}

        .ngram-token {{
            display: inline-flex;
            align-items: center;
            gap: 3px;
            margin: 2px 2px;
            padding: 2px 4px;
            border-radius: 5px;
            background: #f3f4f6;
            border: 1px solid transparent;
            vertical-align: middle;
        }}

        .ngram-token-hit {{
            border: 1px solid rgba(0, 0, 0, 0.35);
            box-shadow: inset 0 0 0 1px rgba(255, 255, 255, 0.25);
            transition: opacity 0.15s ease, filter 0.15s ease, outline 0.15s ease;
        }}

        .ngram-token-index {{
            font-size: 9px;
            color: #6b7280;
            border-right: 1px solid rgba(0, 0, 0, 0.18);
            padding-right: 3px;
        }}

        .ngram-token-text {{
            white-space: pre;
        }}

        .ngram-hit-label {{
            display: inline-block;
            background: rgba(17, 24, 39, 0.85);
            color: white;
            border-radius: 4px;
            padding: 0 3px;
            font-size: 9px;
            font-weight: 700;
            margin-right: 2px;
        }}

        .ngram-legend {{
            margin-bottom: 14px;
            border: 1px solid #d1d5db;
            border-radius: 8px;
            background: white;
            padding: 8px 10px;
        }}

        .ngram-legend summary {{
            cursor: pointer;
            font-weight: 700;
        }}

        .ngram-legend-body {{
            margin-top: 8px;
            display: flex;
            flex-wrap: wrap;
            gap: 6px;
            max-height: 220px;
            overflow: auto;
        }}

        .ngram-legend-chip {{
            display: inline-block;
            border: 1px solid rgba(0, 0, 0, 0.25);
            border-radius: 999px;
            padding: 3px 8px;
            max-width: 360px;
            overflow: hidden;
            text-overflow: ellipsis;
            white-space: nowrap;
            font-size: 12px;
            cursor: pointer;
            user-select: none;
            transition: opacity 0.15s ease, outline 0.15s ease;
        }}

        .ngram-legend-clear {{
            background: #111827;
            color: white;
            font-weight: 700;
        }}

        .ngram-legend-more {{
            color: #6b7280;
            font-style: italic;
            padding: 3px 8px;
        }}

        .ngram-omitted {{
            color: #6b7280;
            font-style: italic;
            padding: 4px 6px;
        }}

        {selection_css}
    </style>

    <div class="ngram-container">
        <div class="ngram-title">{_html_escape(title)}</div>

        <div class="ngram-meta">
            <span class="ngram-pill">N-grams: {len(ngrams)}</span>
            <span class="ngram-pill">Total occurrences: {total_occurrences}</span>
            <span class="ngram-pill">Unknown tokens: {len(unk_tokens)}</span>
            <span class="ngram-pill">Known texts: {len(known_texts)}</span>
            <span class="ngram-pill">Known tokens: {total_known_tokens}</span>
            <span class="ngram-pill">Greatest common mode: {greatest_common}</span>
        </div>

        {legend_html}

        <div class="ngram-layout">
            {unknown_panel}
            {''.join(known_panels)}
        </div>
    </div>
    """


def show_ngram_highlights_pretty(
    *,
    unknown_text: str,
    known_texts: Sequence[str],
    tokenizer: Any,
    ngrams,
    known_doc_ids: Optional[Sequence[Any]] = None,
    unknown_doc_id: Optional[Any] = None,
    lowercase: bool = True,
    greatest_common: bool = False,
    all_ngrams: Optional[Sequence[Sequence[Any]]] = None,
    title: str = "Common token n-gram highlights",
    max_tokens_per_text: Optional[int] = None,
    show_token_indices: bool = True,
    show_start_labels: bool = True,
    max_legend_items: Optional[int] = 200,
):
    """
    Display GST-style n-gram highlights in a notebook.
    """
    out_html = build_ngram_highlights_html(
        unknown_text=unknown_text,
        known_texts=known_texts,
        tokenizer=tokenizer,
        ngrams=ngrams,
        known_doc_ids=known_doc_ids,
        unknown_doc_id=unknown_doc_id,
        lowercase=lowercase,
        greatest_common=greatest_common,
        all_ngrams=all_ngrams,
        title=title,
        max_tokens_per_text=max_tokens_per_text,
        show_token_indices=show_token_indices,
        show_start_labels=show_start_labels,
        max_legend_items=max_legend_items,
    )

    display(HTML(out_html))

In [137]:
show_ngram_highlights_pretty(
    unknown_text=unknown_text,
    known_texts=selected_known["text"].tolist(),
    tokenizer=tokenizer,
    ngrams=global_kept_ngrams,
    all_ngrams=global_kept_ngrams,
    known_doc_ids=selected_known["doc_id"].tolist(),
    unknown_doc_id=unknown_doc,
    lowercase=True,
    greatest_common=True,
    max_tokens_per_text=None,
    max_legend_items=None,
)

In [114]:
def find_parent_ngrams(target_ngram, candidate_ngrams):
    """
    Find longer n-grams in candidate_ngrams that contain target_ngram.
    """
    target = tuple(target_ngram)
    parents = []

    for g in candidate_ngrams:
        g = tuple(g)

        if len(g) <= len(target):
            continue

        for i in range(len(g) - len(target) + 1):
            if g[i:i + len(target)] == target:
                parents.append(list(g))
                break

    return parents

def find_parent_ngrams_with_offsets(target_ngram, candidate_ngrams):
    """
    Find longer n-grams in candidate_ngrams that contain target_ngram.

    Returns:
        [
            {
                "parent_ngram": [...],
                "offset": int
            },
            ...
        ]
    """
    target = tuple(target_ngram)
    parents = []

    for g in candidate_ngrams:
        parent = tuple(g)

        if len(parent) <= len(target):
            continue

        for offset in range(len(parent) - len(target) + 1):
            if parent[offset:offset + len(target)] == target:
                parents.append({
                    "parent_ngram": list(parent),
                    "offset": offset,
                })

    return parents


def print_ngrams_with_parents(result_tokens, *, max_parents=None):
    """
    Print only n-grams from result_tokens that are subgrams of longer n-grams
    also in result_tokens.

    Output includes:
      - the original/child n-gram
      - its parent n-grams
      - a copy-paste block for show_ngram_highlights()
    """
    grams = [
        list(g)
        for g in dict.fromkeys(tuple(g) for g in result_tokens)
    ]

    found_any = False

    for child in grams:
        parents = find_parent_ngrams_with_offsets(child, grams)

        if not parents:
            continue

        found_any = True

        print("\n" + "=" * 80)
        print("CHILD:")
        print(child)

        print("\nPARENTS:")
        shown = parents if max_parents is None else parents[:max_parents]

        for p in shown:
            print(f"offset {p['offset']}: {p['parent_ngram']}")

        if max_parents is not None and len(parents) > max_parents:
            print(f"... plus {len(parents) - max_parents} more")

        print("\nCOPY INTO HIGHLIGHTER:")
        print("[")
        print(f"    {child},")
        for p in shown:
            print(f"    {p['parent_ngram']},")
        print("]")

    if not found_any:
        print("No n-grams have parent n-grams in this list.")
        
def show_ngram_occurrences(
    text,
    target_ngram,
    candidate_ngrams,
    tokenizer,
    *,
    lowercase=True,
    window=12,
    label="text",
):
    """
    Print occurrences of target_ngram, showing whether each occurrence is
    inside a longer candidate n-gram.
    """
    tokens = tokenize_to_tokens(
        text,
        tokenizer=tokenizer,
        lowercase=lowercase,
    )

    target_spans = find_all_token_ngram_spans(
        tokens=tokens,
        ngram_tokens=target_ngram,
        allow_overlaps=True,
    )

    parent_ngrams = find_parent_ngrams(target_ngram, candidate_ngrams)

    parent_spans = []

    for parent in parent_ngrams:
        spans = find_all_token_ngram_spans(
            tokens=tokens,
            ngram_tokens=parent,
            allow_overlaps=True,
        )

        for span in spans:
            parent_spans.append({
                "parent_ngram": parent,
                "span": span,
            })

    print("\n" + "=" * 80)
    print(label)
    print(f"Target n-gram: {target_ngram}")
    print(f"Target occurrences: {len(target_spans)}")
    print(f"Number of possible parent n-grams: {len(parent_ngrams)}")

    if parent_ngrams:
        print("\nParent n-grams containing target:")
        for p in parent_ngrams[:20]:
            print(" ", p)

        if len(parent_ngrams) > 20:
            print(f"  ... plus {len(parent_ngrams) - 20} more")

    print("\nOccurrences:")

    for occ_i, (s, e) in enumerate(target_spans, start=1):
        left = max(0, s - window)
        right = min(len(tokens), e + window)

        context_tokens = tokens[left:right]
        context_text = tokens_to_text(context_tokens, tokenizer)

        blockers = [
            p
            for p in parent_spans
            if p["span"][0] <= s and e <= p["span"][1]
        ]

        print("\n---")
        print(f"Occurrence {occ_i}: span=({s}, {e})")
        print(f"Blocked by longer n-gram? {len(blockers) > 0}")
        print(f"Context: {context_text}")

        if blockers:
            print("Blocking parent spans:")
            for b in blockers[:10]:
                print(f"  span={b['span']} parent={b['parent_ngram']}")

            if len(blockers) > 10:
                print(f"  ... plus {len(blockers) - 10} more")

In [55]:
# import ast

# data = read_excel_sheets(file_loc, sheet_names=["phrase score"])
# data = data['phrase score']

# result_tokens = data["tokens"].apply(ast.literal_eval).tolist()

# print(f"Number of Tokens in Result: {len(result_tokens)}")

In [69]:
result_tokens = current_comparison['kept']
# print_ngrams_with_parents(result_tokens)

In [65]:
current_removed

[['Ġit'], ['Ġat'], ['Ġhave'], ['Ġthere'], ['Ġall', 'Ġthe']]

In [115]:
n_gram_for_parent = ['Ġall', 'Ġthe']
parent_list = find_parent_ngrams(n_gram_for_parent, result_tokens)

full_parent_list = [n_gram_for_parent] + parent_list
print(full_parent_list)

n_gram_to_check = (
full_parent_list
)

show_ngram_highlights(
    unknown_text=unknown_text,
    known_texts=selected_known["text"].tolist(),
    known_doc_ids=selected_known["doc_id"].tolist(),
    unknown_doc_id=unknown_doc,
    tokenizer=tokenizer,
    ngrams=result_tokens,
    lowercase=True,
    greatest_common=True,
    all_ngrams=result_tokens,
)

[['Ġall', 'Ġthe'], ['Ġof', 'Ġall', 'Ġthe']]
